# Reward 模型

CartPole、LunarLander 等游戏里面他的奖励由环境反馈

但是在具体的LLM问题中，当前决策的好坏并没有直接的反馈， 因此需要一个Reward模型反馈

# **Badly Badly Terror 模型**

对于一个问题 \(x\)，假设有两个回答：

- \(y_w\)：更好的回答，即 winner；
- \(y_l\)：较差的回答，即 loser。

奖励模型 \(r(x,y)\) 会针对问题 \(x\) 和回答 \(y\) 输出一个实数分数。我们希望：

$$
r(x,y_w)>r(x,y_l).
$$

Bradley–Terry 模型将“回答 \(y_w\) 优于回答 \(y_l\)”的概率定义为：

$$
P(y_w>y_l\mid x)
=\sigma\left(r(x,y_w)-r(x,y_l)\right),
$$

其中 Sigmoid 函数为：

$$
\sigma(z)=\frac{1}{1+e^{-z}}.
$$

---

**为什么使用减法？**

定义两个回答的奖励差：

$$
\Delta r=r(x,y_w)-r(x,y_l).
$$

模型只需要判断两个回答的相对优劣，而不需要关心奖励分数的绝对大小：

- 当 \(\Delta r>0\) 时，模型认为 \(y_w\) 更好；
- 当 \(\Delta r=0\) 时，模型认为两个回答同样好；
- 当 \(\Delta r<0\) 时，模型错误地认为 \(y_l\) 更好。

使用差值还有一个重要性质：同时给两个回答的奖励增加同一个常数 \(C\)，比较结果不会发生变化：

$$
\begin{aligned}
&[r(x,y_w)+C]-[r(x,y_l)+C]\\
={}&r(x,y_w)-r(x,y_l).
\end{aligned}
$$

因此，Bradley–Terry 模型学习的是回答之间的**相对偏好**，而不是奖励分数的绝对标尺。

---

**为什么使用 Sigmoid？**

奖励差 \(\Delta r\) 的取值范围是整个实数轴：

$$
\Delta r\in(-\infty,+\infty).
$$

Sigmoid 可以将它转换为 \((0,1)\) 之间的概率：

$$
\sigma(\Delta r)\in(0,1).
$$

具体来说：

$$
\begin{cases}
\Delta r\gg 0, & \sigma(\Delta r)\approx 1,\\
\Delta r=0, & \sigma(\Delta r)=0.5,\\
\Delta r\ll 0, & \sigma(\Delta r)\approx 0.
\end{cases}
$$

因此：

- 奖励差越大，模型认为 \(y_w\) 更好的概率越高；
- 奖励差为零，模型认为两个回答各有 \(50\%\) 的胜率；
- 奖励差为负，模型认为 \(y_w\) 更好的概率低于 \(50\%\)。

Sigmoid 还是一个单调、连续且可微的函数，因此适合通过梯度下降训练神经网络。

> Sigmoid 并不意味着模型“完全不受数值大小影响”。当输入的绝对值很大时，Sigmoid 会逐渐饱和，梯度也会变小。它的主要作用是将奖励差解释成概率，并提供可微的优化目标。

---

**奖励模型的损失函数**

我们希望偏好数据中标记的 winner \(y_w\) 获得尽可能高的胜出概率：

$$
P(y_w>y_l\mid x)
=\sigma(\Delta r).
$$

因此可以使用最大似然估计，最大化：

$$
\log\sigma(\Delta r).
$$

训练通常采用梯度下降，所以将最大化问题改写为最小化负对数似然：

$$
\boxed{
\mathcal L
=-\log\sigma\left(r(x,y_w)-r(x,y_l)\right)
}
$$

这里不是简单地在 Sigmoid 前面加负号，而是使用：

$$
-\log\sigma(\Delta r).
$$

原因是：

- 当 \(\Delta r\) 很大时，\(\sigma(\Delta r)\approx1\)，损失接近 \(0\)；
- 当 \(\Delta r=0\) 时，\(\sigma(\Delta r)=0.5\)，损失为 \(-\log 0.5\)；
- 当 \(\Delta r\) 很小时，\(\sigma(\Delta r)\approx0\)，损失会变得很大。

因此，最小化这个损失会推动：

$$
r(x,y_w)-r(x,y_l)
$$

不断增大，也就是推动模型满足：

$$
r(x,y_w)>r(x,y_l).
$$

In [ ]:
import torch
import torch.nn as nn 

class RewardModel(nn.Module):
    """奖励模型：输入 (prompt, response)，输出标量分数"""
    def __init__(self, base_model, hidden_dim=1024):
        super().__init__()
        # 基座模型（通常用 SFT 模型或较小的预训练模型）
        self.base = base_model
        # 在最后一层隐藏状态上加一个线性头，输出标量
        self.reward_head = nn.Linear(hidden_dim, 1)
        
    def forward(self, input_ids, attention_mask):
        # 取最后一个 token 的隐藏状态
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state[:, -1, :]  # (batch, hidden_dim)
        reward = self.reward_head(last_hidden)  # (batch, 1)
        return reward.squeeze(-1)   


def bradley_terry_loss(rm, chosen_ids, chosen_mask, rejected_ids, rejected_mask):
    """Bradley-Terry 偏好损失"""
    r_chosen = rm(chosen_ids, chosen_mask)     # chosen 回答的分数
    r_rejected = rm(rejected_ids, rejected_mask)  # rejected 回答的分数

    # 核心：让 chosen 的分数比 rejected 高
    loss = -torch.nn.functional.logsigmoid(r_chosen - r_rejected).mean()
    return loss    
    

# Reward 评价大模型的指标
这里reward model评价一个大模型往往有几个参数


其中reward model 有三种评分细则：
1. Sequence reward 就是对整句话进行一个奖励评分
2. Step Reward，因为它可以对每个 Prompt 的回答中的每一步推理都进行反馈。其中reasoning steps，是按照回答中的句号或回车等标点符号来进行的。
3. token Reward 是最少用的，因为它划分得太细，而且也很难说明它每个 token 是不是对的。

In [1]:
# ==========================================
# 不同粒度的奖励计算
# ==========================================
def sequence_reward(rm, prompt, response):
    """Sequence-level：整个回答一个分数"""
    return rm.score(prompt, response)  # 一个标量

def step_reward(rm, prompt, reasoning_steps):
    """Step-level：每个推理步骤一个分数"""
    step_rewards = []
    for i, step in enumerate(reasoning_steps):
        # 对前 i+1 步的累积内容打分
        partial = "\n".join(reasoning_steps[:i+1])
        step_rewards.append(rm.score(prompt, partial))
    return step_rewards  # 一个列表

def combined_reward(rm, prompt, response, reasoning_steps):
    """混合：sequence-level RM + step-level 规则"""
    r_rm = sequence_reward(rm, prompt, response)
    r_format = 0.2 if validate_format(response) else 0.0  # 格式奖励
    r_correct = 1.0 if check_answer_correct(prompt, response) else 0.0  # 正确性奖励
    r_length = -0.01 * max(0, len(response) - 500)  # 长度惩罚

    return r_rm + r_format + r_correct + r_length

# 评价Reward模型的指标

下面length_corr 计算的是皮尔逊相关系数：

$$\begin{pmatrix}  \text{rewards与rewards的相关性} & \text{rewards与lengths的相关性} \\ \text{lengths与rewards的相关性} & \text{lengths与lengths的相关性} \end{pmatrix}$$

然后取[0,1]相当于是rewards 和 lengths 之间的相关性 $r$。这个值存放在矩阵的行索引 0、列索引 1

如果该值很高（比如 $> 0.6$）：说明你的 RM 严重偏心长回答。哪怕 Rejected 回答（废话连篇但字多）语义很烂，RM 也会因为它字多而给高分。这就说明你的 RM 被“字数”给绑架了。如果该值很低（比如 $0.1 \sim 0.3$ 左右）：说明 RM 并没有盲目崇拜长文本，而是真正根据回答的逻辑、准确性来打分的。这才是我们想要的高质量奖励模型。

In [4]:
def rm_eval_metrics(r_chosen, r_rejected, chosen_lengths, rejected_lengths):
    import numpy as np

    margin = np.asarray(r_chosen) - np.asarray(r_rejected)
    accuracy = float((margin > 0).mean())

    rewards = np.concatenate([r_chosen, r_rejected])
    lengths = np.concatenate([chosen_lengths, rejected_lengths])
    length_corr = float(np.corrcoef(rewards, lengths)[0, 1])

    return {
        "pairwise_accuracy": accuracy,
        "mean_margin": float(margin.mean()),
        "median_margin": float(np.median(margin)),
        "length_reward_corr": length_corr,
    }